### Semantic chunking
- semantic chunker is a document splitter that uses embedding similarity between sentences to decide chunk boundaries.
- it ensures that chunk is semantically coherent and not cut off mid-thought like traditional character/token splitters.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [8]:
##intialize the model
model= SentenceTransformer('all-MiniLm-L6-v2')

text ="""
Langchain is a frame work for building application with LLMs.
Langchain provides modular abstractions to combine LLms with tools.
Artificial Intelligence is transforming industries by automating repetitive tasks and enabling predictive analytics. 
Artificial Intelligence is to analyze medical images, while banks rely on it to detect fraud.
Private companies like SpaceX and Blue Origin are competing to make space travel more affordable,while NASA continues its mission to explore Mars or space.
"""

## split it into sentences
sentences=[s.strip() for s in text.split("\n") if s.strip()]

### embed each sentence
embedded = model.encode(sentences)

##initalize the parameters
threshold = 0.7
chunks = []
current_chunks =[sentences[0]]

##semantic grouping based on threshold

for i in range(1, len(sentences)):
    similarity = cosine_similarity(
        [embedded[i - 1]],
        [embedded[i]]
    )[0][0]
    
    if similarity>=threshold:
        current_chunks.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunks))
        current_chunks=[sentences[i]]
        
chunks.append(" ".join(current_chunks))

# output
print("\n semantic chunks: ")
for idx,chunk in enumerate(chunks):
    print(f"\n chunks{idx+1}:\n{chunk}")




 semantic chunks: 

 chunks1:
Langchain is a frame work for building application with LLMs. Langchain provides modular abstractions to combine LLms with tools.

 chunks2:
Artificial Intelligence is transforming industries by automating repetitive tasks and enabling predictive analytics.

 chunks3:
Artificial Intelligence is to analyze medical images, while banks rely on it to detect fraud.

 chunks4:
Private companies like SpaceX and Blue Origin are competing to make space travel more affordable,while NASA continues its mission to explore Mars or space.


## RAG pipeline in modular coding

In [9]:
from sentence_transformers import SentenceTransformer 
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableMap
from langchain_core.output_parsers import StrOutputParser
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [ ]:
### Custom semantic chunker with threshold

class ThresholdssemanticChunker:
    def __init__(self,model_name="all-MiniLm-L6-v2",threshold=0.7):
        self.model = SentenceTransformer(model_name)
        self.thresholod= threshold
    
    def split(self,text:str):
        sentences =[s.strip() for s in text.split('.') if s.strip()]
        embedded =self.model.encode(sentences)
        chunks =[]
        current_chunks =[sentences[0]]
        
        for i in range(1,len(sentences)):
            sim = cosine_similarity([embedded[i -1]],[embedded[i]])[0][0]
            if sim>=threshold:
                current_chunks.append(sentences[i])
            else:
                chunks.append(" ".join(current_chunks))
                current_chunks=[sentences[i]]
            
        chunks.append(".".join(current_chunks)+".")
        return chunks
    def split_documents(self,docs):
        res =[]
        for doc in docs:
            for chunk in self.split(doc.page_content):
                res.append(Document(page_content=chunk,metadata=doc.metadata))
        return res

In [13]:

text ="""
Langchain is a frame work for building application with LLMs.
Langchain provides modular abstractions to combine LLms with tools.
Artificial Intelligence is transforming industries by automating repetitive tasks and enabling predictive analytics. 
Artificial Intelligence is to analyze medical images, while banks rely on it to detect fraud.
Private companies like SpaceX and Blue Origin are competing to make space travel more affordable,while NASA continues its mission to explore Mars or space.
"""
doc = Document(page_content=text)
doc

Document(metadata={}, page_content='\nLangchain is a frame work for building application with LLMs.\nLangchain provides modular abstractions to combine LLms with tools.\nArtificial Intelligence is transforming industries by automating repetitive tasks and enabling predictive analytics. \nArtificial Intelligence is to analyze medical images, while banks rely on it to detect fraud.\nPrivate companies like SpaceX and Blue Origin are competing to make space travel more affordable,while NASA continues its mission to explore Mars or space.\n')

In [15]:
##chunking
chunker = ThresholdssemanticChunker(threshold=0.7)
chunks=chunker.split_documents([doc])
chunks

[Document(metadata={}, page_content='Langchain is a frame work for building application with LLMs Langchain provides modular abstractions to combine LLms with tools'),
 Document(metadata={}, page_content='Artificial Intelligence is transforming industries by automating repetitive tasks and enabling predictive analytics'),
 Document(metadata={}, page_content='Artificial Intelligence is to analyze medical images, while banks rely on it to detect fraud'),
 Document(metadata={}, page_content='Private companies like SpaceX and Blue Origin are competing to make space travel more affordable,while NASA continues its mission to explore Mars or space.')]

In [18]:
##vector store

embedding = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
Vectrostore=FAISS.from_documents(chunks,embedding)
retriever = Vectrostore.as_retriever()
retriever


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000023886D32AD0>, search_kwargs={})

In [20]:
## prompt template
template = """ Answer the Questions based on the conteext:
{context}

Question:{Question}
"""
prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['Question', 'context'], input_types={}, partial_variables={}, template=' Answer the Questions based on the conteext:\n{context}\n\nQuestion:{Question}\n')

In [30]:
## llm

llm = init_chat_model(model="groq/compound-mini", model_provider="groq", temperature=0.4)

### lcel
# PromptTemplate expects input variables named 'Question' and 'context'
# so ensure the RunnableMap produces those exact keys and the query uses 'Question'
rag_chain =(
    RunnableMap(
    {
    "context": lambda x: retriever.invoke(x["Question"]),
    "Question": lambda x: x["Question"],
    })
    | prompt
    | llm 
    | StrOutputParser()
)

query = {"Question": "which is to analyze medical images?"}
result = rag_chain.invoke(query)
print(result)


Artificial Intelligence.
